In [12]:
import numpy as np
from scipy.stats import norm
import plotly.graph_objects as go

The Black-Scholes model provides a closed-form solution for pricing European call options under the assumption that the underlying asset follows a GBM.


C = S₀ N(d₁) - K e^{-rT} N(d₂)

In [3]:
def black_scholes_call(S, K, T, r, sigma):
    d1= (np.log(S/K)+(r+sigma**2/2)*T)/(sigma*np.sqrt(T))
    d2=d1-(sigma*np.sqrt(T))
    N_d1=norm.cdf(d1)
    N_d2=norm.cdf(d2)
    C= S * N_d1-K * np.exp(-r*T) * N_d2
    return C

In [4]:
S = 100
K = 100
T = 1
r = 0.05
sigma = 0.2

print(black_scholes_call(S, K, T, r, sigma))

10.450583572185565


Monte Carlo simulation estimates the same expected payoff numerically by simulating terminal stock prices under GBM.
So, if my implementation is correct they should both coverge to 10.4....

In [32]:
def monte_carlo_call(S, K, T, r, sigma,num=10000):
    np.random.seed(42)
    Z=np.random.normal(0,1,num)
    S_t=S*np.exp(((r-(sigma**2/2))*T)+(sigma*np.sqrt(T)*Z))
    payoff=np.maximum(S_t-K,0)
    price=np.exp(-r*T)*np.mean(payoff)
    return price


In [33]:
print(monte_carlo_call(100, 100, 1, 0.05, 0.2))

10.450169921134655


In [34]:
sigma_values=np.linspace(0.01,0.1,50)
black_scholes_prices=[]
monte_carlo_prices=[]

In [35]:
for sigma in sigma_values:
    bs = black_scholes_call(100, 100, 1, 0.05, sigma)
    mc = monte_carlo_call(100, 100, 1, 0.05, sigma)

    black_scholes_prices.append(bs)
    monte_carlo_prices.append(mc)

In [36]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sigma_values,
    y=black_scholes_prices,
    mode='lines',
    name='Black-Scholes'
))
fig.add_trace(go.Scatter(
    x=sigma_values,
    y=monte_carlo_prices,
    mode='markers',
    name='Monte Carlo'
))
fig.update_layout(
    title='Option Price vs Volatility',
    xaxis_title='Volatility (sigma)',
    yaxis_title='Option Price'
)